[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab03_supervised_unsupervised_rl.ipynb)

# Lab 3: การเรียนรู้แบบมีผู้สอน การเรียนรู้แบบไม่มีผู้สอน และการเรียนรู้แบบเสริมกำลัง (Supervised, Unsupervised & Reinforcement Learning)

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันอังคารที่ 6 ตุลาคม 2026 · **เวลาโดยประมาณ:** 60–90 นาที

## วัตถุประสงค์การเรียนรู้

- ฝึกสอนและเปรียบเทียบตัวจำแนกประเภทแบบมีผู้สอน (supervised classifiers) ได้แก่ logistic regression, decision tree, random forest และ XGBoost
- ประยุกต์ใช้การเรียนรู้แบบไม่มีผู้สอน (unsupervised learning) ได้แก่ การจัดกลุ่มด้วย K-means และการวิเคราะห์องค์ประกอบหลัก (PCA) เพื่อจำแนกกลุ่มผู้ป่วย (patient stratification)
- อธิบายกรอบแนวคิดของการเรียนรู้แบบเสริมกำลัง (reinforcement learning, RL) ผ่านตัวอย่างนโยบายการรักษา (treatment policy) อย่างง่าย
- เลือกรูปแบบการเรียนรู้ (learning paradigm) ให้เหมาะสมกับโจทย์ทางคลินิก
- **แปลผล**สิ่งที่แต่ละโมเดลเรียนรู้ และประเมินความสมเหตุสมผลทางคลินิก

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ ทางซ้ายของเซลล์ หรือใช้ **Shift+Enter**) เซลล์ ⭐ **Exercise** แต่ละเซลล์กำหนดให้แก้ไขค่าเพียงหนึ่งค่าหรือเติมโค้ดสั้น ๆ หนึ่งบรรทัด และมีเซลล์ ✅ **เฉลย** อยู่ถัดไป ผู้เรียนควรทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำส่วนถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความ (ดับเบิลคลิกที่เซลล์ `✍️ คำตอบ:` เพื่อพิมพ์) ซึ่งถือเป็น**ชิ้นงานที่ต้องส่ง** คำถามมุ่งเน้นการแปลผลว่า*โมเดลพบสิ่งใด และสมเหตุสมผลทางคลินิกหรือไม่* มิใช่การเขียนโค้ด
>
> 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** ส่วนท้ายของโน้ตบุ๊ก (notebook) มีโจทย์ท้าทายซึ่งให้คะแนนอัตโนมัติบน[ตารางอันดับ (leaderboard) ของรายวิชาบน Hugging Face](https://huggingface.co/spaces/wdittaya/aidh-leaderboard) ส่วนนี้**ไม่บังคับ**และไม่นับเป็นชิ้นงานที่ต้องส่ง ผู้เรียนที่ประสงค์จะเข้าร่วมให้ระบุนามแฝงใน `NAME` และรหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom ใน `JOIN_CODE` ในเซลล์การตั้งค่าด้านล่าง
>
> ปฏิบัติการนี้ใช้ [Google Colab](https://colab.research.google.com) รุ่นไม่มีค่าใช้จ่ายได้ภายในเวลาไม่กี่นาที (ไม่จำเป็นต้องใช้ GPU) หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
!pip -q install gradio_client huggingface_hub

HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
LAB_ID    = "lab03"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

from huggingface_hub import hf_hub_download

def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    from gradio_client import Client, handle_file
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    client = Client(SPACE_ID, verbose=False)
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ชุดข้อมูล (dataset) — UCI Heart Disease (Cleveland)

ชุดข้อมูลประกอบด้วยผู้ป่วย 303 รายที่ได้รับการส่งตัวเพื่อตรวจสวนหลอดเลือดหัวใจ (coronary angiography) ที่ Cleveland Clinic มีตัวแปรทางคลินิก 13 ตัว และตัวแปรเป้าหมาย (target) คือการมีโรคหลอดเลือดหัวใจ (หลอดเลือดหัวใจหลักตีบตั้งแต่ 50 % ขึ้นไปจากการสวนหัวใจ) เป็นข้อมูลสาธารณะจาก UCI เมื่อตัดผู้ป่วย 6 รายที่มีข้อมูลขาดหาย (missing values) ออก จะเหลือผู้ป่วย 297 ราย

| คอลัมน์ | ความหมาย |
|---|---|
| `age` | อายุ (ปี) |
| `sex` | 1 = ชาย, 0 = หญิง |
| `cp` | ลักษณะอาการเจ็บหน้าอก: 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = ไม่มีอาการ (asymptomatic) |
| `trestbps` | ความดันโลหิตขณะพัก (mmHg) |
| `chol` | ระดับคอเลสเตอรอลในเลือด (mg/dL) |
| `fbs` | ระดับน้ำตาลในเลือดขณะอดอาหาร > 120 mg/dL (1 = ใช่) |
| `restecg` | คลื่นไฟฟ้าหัวใจขณะพัก: 0 = ปกติ, 1 = ST-T ผิดปกติ, 2 = หัวใจห้องล่างซ้ายโต (LVH) |
| `thalach` | อัตราการเต้นของหัวใจสูงสุดระหว่างการทดสอบการออกกำลังกาย (exercise test) (ครั้ง/นาที) |
| `exang` | อาการเจ็บหน้าอกขณะออกกำลังกาย (exercise-induced angina) (1 = ใช่) |
| `oldpeak` | ST depression ขณะออกกำลังกายเทียบกับขณะพัก (มม.) |
| `slope` | ความชันของ ST segment ช่วงออกกำลังกายสูงสุด: 1 = ขึ้น, 2 = ราบ, 3 = ลง |
| `ca` | จำนวนหลอดเลือดหลัก (0–3) ที่ปรากฏจากการฉีดสี (fluoroscopy) |
| `thal` | ผลการตรวจ thallium stress scan: 3 = ปกติ, 6 = fixed defect, 7 = reversible defect |
| `target` | **1 = เป็นโรคหัวใจ, 0 = ไม่เป็นโรคหัวใจ** (ตัวแปรที่ต้องการทำนาย) |

ข้อควรทราบ: `cp`, `restecg`, `slope` และ `thal` เป็น**รหัสหมวดหมู่ (category codes)** มิใช่ค่าที่ได้จากการวัด เช่น "`thal` = 7" มิได้มีค่า "มากกว่า" 6 ในความหมายทางกายภาพ ผู้เรียนควรคำนึงถึงข้อนี้ในการแปลผลโมเดล

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

cols = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","target"]
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
df = pd.read_csv(URL, names=cols, na_values="?")
df["target"] = (df["target"] > 0).astype(int)  # แปลงเป็นสองกลุ่ม: เป็นโรค / ไม่เป็นโรค
df = df.dropna().reset_index(drop=True)
print(df.shape, f"- ผู้ป่วยร้อยละ {df['target'].mean():.0%} เป็นโรคหัวใจ")
df.head()

> 💡 **คำอธิบายผลลัพธ์** เซลล์นี้ดาวน์โหลดตารางข้อมูล (หนึ่งแถวต่อผู้ป่วยหนึ่งราย) และแปลงระดับโรคเดิม 0–4 ให้เป็นป้ายกำกับ (label) แบบสองค่า: `target` = 1 หากพบหลอดเลือดหัวใจตีบอย่างมีนัยสำคัญ ผู้ป่วยประมาณครึ่งหนึ่งเป็นโรค ทั้งสองกลุ่มจึงมีขนาดใกล้เคียงกัน

## Part A — การเรียนรู้แบบมีผู้สอน (supervised learning): การทำนายโรคหัวใจ

**การเรียนรู้แบบมีผู้สอน** คือการเรียนรู้จากตัวอย่างที่ทราบคำตอบแล้ว (เรียกว่า*ป้ายกำกับ (label)* ในที่นี้คือ `target`) เปรียบได้กับแพทย์ประจำบ้านที่เรียนรู้จากผู้ป่วยซึ่งได้รับการวินิจฉัยยืนยันแล้ว

ผู้ป่วยร้อยละ 25 ถูกแยกไว้เป็น*ชุดทดสอบ (test set)* และเปรียบเทียบโมเดล 4 ประเภท ดังนี้
- **Logistic regression**: คะแนนความเสี่ยงแบบดั้งเดิม โดยกำหนดน้ำหนักให้แต่ละตัวแปร แล้วรวมผลเป็นความน่าจะเป็น
- **Decision tree (ต้นไม้ตัดสินใจ)**: ผังคำถามใช่/ไม่ใช่ที่ต่อกันเป็นลำดับ ("`thal` ปกติหรือไม่ → `ca` = 0 หรือไม่ → …")
- **Random forest**: ต้นไม้ตัดสินใจหลายร้อยต้นที่ลงคะแนนร่วมกัน
- **XGBoost**: สร้างต้นไม้ทีละต้น โดยแต่ละต้นแก้ไขความคลาดเคลื่อนของต้นก่อนหน้า (*gradient boosting*)

การประเมินโมเดลใช้ **ROC AUC** ซึ่งหมายถึงความน่าจะเป็นที่โมเดลให้ค่าความเสี่ยงแก่ผู้ป่วยที่*เป็น*โรค (สุ่มมา 1 ราย) สูงกว่าผู้ป่วยที่*ไม่เป็น*โรค (สุ่มมา 1 ราย) ค่า 0.5 เทียบเท่าการเดาสุ่ม และค่า 1.0 คือการจำแนกได้สมบูรณ์ เพื่อให้ได้ค่าประมาณที่เป็นธรรม จึงใช้**การตรวจสอบไขว้ 5 ส่วน (5-fold cross-validation, CV)** กล่าวคือ แบ่งผู้ป่วยชุดฝึกเป็น 5 ส่วน ฝึกด้วย 4 ส่วนและทดสอบด้วยส่วนที่เหลือ หมุนเวียนจนครบทุกส่วน แล้วหาค่าเฉลี่ย

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

X = df.drop(columns="target")
y = df["target"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

models = {
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "DecisionTree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
}

def compare_models(models):
    # ค่า ROC AUC จากการตรวจสอบไขว้ 5 ส่วนของทุกโมเดลใน dict เรียงจากมากไปน้อย
    cv_auc = {name: cross_val_score(m, X_tr, y_tr, cv=5, scoring="roc_auc").mean()
              for name, m in models.items()}
    return pd.Series(cv_auc, name="CV ROC AUC").sort_values(ascending=False).round(3)

compare_models(models)

> 💡 **คำอธิบายผลลัพธ์** แต่ละโมเดลได้รับการฝึกและทดสอบ 5 รอบบนส่วนต่าง ๆ ของผู้ป่วยชุดฝึก 222 ราย ตารางแสดงค่า ROC AUC เฉลี่ย สำหรับชุดข้อมูลขนาดเล็กเช่นนี้ ความแตกต่างประมาณ 0.01–0.02 ถือว่าอยู่ในระดับความแปรปรวนตามปกติ (noise)

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: เพิ่ม XGBoost ในการเปรียบเทียบ
# XGBoost เป็นโมเดลประเภท "boosted trees" ที่ใช้กันแพร่หลาย บรรทัดด้านล่างยังขาดการกำหนดค่าของโมเดล
# สิ่งที่ต้องทำ: แทนที่ `...` ด้วย
#     XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.1, random_state=42, eval_metric="logloss")
# (n_estimators = จำนวนต้นไม้, max_depth = จำนวนคำถามสูงสุดของต้นไม้แต่ละต้น,
#  learning_rate = ระดับที่ต้นไม้ต้นใหม่แก้ไขความคลาดเคลื่อนของต้นก่อนหน้า)
# คำแนะนำ: คัดลอกข้อความข้างต้นให้ถูกต้องทุกตัวอักษร แล้วรันเซลล์
!pip -q install "xgboost>=2.1.4"
from xgboost import XGBClassifier

models["XGBoost"] = ...   # TODO

compare_models(models)

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
!pip -q install "xgboost>=2.1.4"
from xgboost import XGBClassifier

models["XGBoost"] = XGBClassifier(n_estimators=300, max_depth=3, learning_rate=0.1,
                                  random_state=42, eval_metric="logloss")
compare_models(models)

✍️ **คำถาม A1** โปรดสรุปค่า CV ROC AUC ของโมเดลทั้ง 4 เป็นตาราง (โมเดล → AUC) โมเดลใดให้ผลดีที่สุด โมเดลที่ซับซ้อนกว่า (XGBoost) ให้ผลดีกว่าคะแนนความเสี่ยงอย่างง่าย (logistic regression) อย่างชัดเจนหรือไม่ และโมเดลใดอธิบายต่อผู้ป่วยหรือทีมผู้ให้การรักษาได้ง่ายที่สุด

✍️ **คำตอบ:**

### ตัวแปรใดมีความสำคัญ

ในบริบททางการแพทย์ ความแม่นยำของการทำนายเพียงอย่างเดียวยังไม่เพียงพอ จำเป็นต้องทราบด้วยว่า*โมเดลอาศัยข้อมูลใด* ส่วนนี้พิจารณา 2 มุมมอง ดังนี้
1. **ความสำคัญของตัวแปร (feature importance) จาก random forest**: ระดับที่ตัวแปรแต่ละตัวช่วยให้ต้นไม้แยกผู้ป่วยที่เป็นโรคออกจากผู้ที่ไม่เป็นโรค (ค่ามาก = มีประโยชน์มาก แต่ไม่ระบุทิศทาง)
2. **สัมประสิทธิ์ (coefficients) ของ logistic regression**: น้ำหนักของแต่ละตัวแปรในคะแนนความเสี่ยง พร้อม**เครื่องหมาย** ค่าบวก = ค่าตัวแปรสูงขึ้น ความเสี่ยงที่ทำนายสูงขึ้น, ค่าลบ = ค่าตัวแปรสูงขึ้น ความเสี่ยงที่ทำนายต่ำลง

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: ความสำคัญของตัวแปรใน random forest
# random forest ได้รับการฝึกไว้แล้ว ให้นำค่าความสำคัญของตัวแปรมาสร้างตารางและแผนภูมิ
# สิ่งที่ต้องทำ: แทนที่ `...` ด้วย   rf.feature_importances_
# คำแนะนำ: พิมพ์ "rf." ตามด้วยชื่อ attribute ส่วนที่เหลือของเซลล์จะเรียงลำดับและสร้างแผนภูมิโดยอัตโนมัติ
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_tr, y_tr)

importances = pd.Series(..., index=X.columns)   # TODO

importances.sort_values().plot.barh(figsize=(6, 4), title="Random forest feature importance")
plt.xlabel("importance"); plt.show()

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_tr, y_tr)

importances = pd.Series(rf.feature_importances_, index=X.columns)

importances.sort_values().plot.barh(figsize=(6, 4), title="Random forest feature importance")
plt.xlabel("importance"); plt.show()

In [ ]:
# น้ำหนักของ logistic regression (ตัวแปรผ่านการปรับมาตรฐานแล้ว จึงเปรียบเทียบขนาดกันได้)
# ข้อความในแผนภูมิเป็นภาษาอังกฤษ เนื่องจากฟอนต์เริ่มต้นของ matplotlib บน Colab ไม่รองรับภาษาไทย
logreg = models["LogReg"].fit(X_tr, y_tr)
coef = pd.Series(logreg[-1].coef_[0], index=X.columns).sort_values()
coef.plot.barh(figsize=(6, 4), color=np.where(coef > 0, "tab:red", "tab:blue"),
               title="Logistic regression: red = raises risk, blue = lowers risk")
plt.axvline(0, color="k", lw=0.8); plt.xlabel("coefficient (per 1 SD)"); plt.show()

> 💡 **คำอธิบายผลลัพธ์** แผนภูมิทั้งสองจัดอันดับตัวแปร ในแผนภูมิ logistic regression สีแดงหมายถึงเพิ่มความเสี่ยง และสีน้ำเงินหมายถึงลดความเสี่ยง ส่วน "per 1 SD" หมายถึงต่อหนึ่งส่วนเบี่ยงเบนมาตรฐาน (standard deviation) ซึ่งเป็นขนาดความแตกต่างโดยทั่วไประหว่างผู้ป่วย จึงเปรียบเทียบข้ามตัวแปรที่มีหน่วยต่างกันได้ ทั้งนี้ แผนภูมิแสดงเฉพาะสิ่งที่โมเดล*ใช้* ซึ่งเป็น**ความสัมพันธ์ (association) ในชุดข้อมูลนี้ มิใช่ความเป็นเหตุเป็นผล (causation)**

✍️ **คำถาม A2** โปรดระบุ**ตัวแปรที่มีความสำคัญสูงสุด 3 อันดับแรก**ของ random forest แผนภูมิทั้งสองสอดคล้องกันโดยรวมหรือไม่ และจงอธิบายเชิงคลินิกโดยอ้างอิงตารางคอลัมน์ข้างต้นว่า เหตุใดตัวแปรเหล่านี้จึงสามารถทำนายโรคหลอดเลือดหัวใจได้

✍️ **คำตอบ:**

✍️ **คำถาม A3** จากแผนภูมิ logistic regression ให้พิจารณา*ทิศทาง*ของ `thalach` (อัตราการเต้นของหัวใจสูงสุด) และ `oldpeak` (ST depression) ว่าสอดคล้องกับหลักการของการทดสอบสมรรถภาพหัวใจด้วยการออกกำลังกาย (exercise stress test) หรือไม่ นอกจากนี้ `fbs` (ระดับน้ำตาลในเลือดขณะอดอาหารสูง) และ `chol` (คอเลสเตอรอล) เป็นปัจจัยเสี่ยงที่ทราบกันดี โมเดลจัดอันดับตัวแปรทั้งสองไว้อย่างไร และ `fbs` มีทิศทางใด จงเสนอเหตุผลที่เป็นไปได้ (พิจารณาเกณฑ์การส่งตัวผู้ป่วยมาตรวจสวนหัวใจ ขนาดตัวอย่างที่มีจำกัด และการที่ตัวแปรซึ่งมีความสัมพันธ์กัน "แบ่งน้ำหนัก" ร่วมกัน)

✍️ **คำตอบ:**

## Part B — การเรียนรู้แบบไม่มีผู้สอน (unsupervised learning): การจำแนกกลุ่มผู้ป่วย

**การเรียนรู้แบบไม่มีผู้สอน** คือการค้นหาโครงสร้างในข้อมูล*โดยไม่ใช้*ป้ายกำกับ ส่วนนี้ตั้งคำถามว่า *ผู้ป่วยสามารถแบ่งออกเป็นกลุ่มตามลักษณะที่เกิดขึ้นเองตามธรรมชาติ (phenotypes) ได้หรือไม่* โดยอัลกอริทึมไม่ได้ใช้คอลัมน์ `target` เลย

- **การวิเคราะห์องค์ประกอบหลัก (principal component analysis, PCA)** ลดตัวแปร 13 ตัวให้เหลือแกนสรุป 2 แกน (PC1, PC2) เพื่อแสดงผู้ป่วยแต่ละรายเป็นจุดบนแผนภูมิ
- **K-means** แบ่งผู้ป่วยออกเป็น *K* กลุ่ม (clusters) ตามความคล้ายคลึงกัน

ก่อนการวิเคราะห์ ต้อง*ปรับมาตรฐาน (standardise)* ตัวแปรให้อยู่ในมาตราเดียวกัน มิฉะนั้นคอเลสเตอรอล (≈ 250) จะมีอิทธิพลเหนือ `ca` (0–3)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit_transform(Xs)

plt.scatter(pca[:,0], pca[:,1], c=y, cmap="coolwarm", alpha=0.7)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Patients in PCA space (coloured by true label: red = disease)")
plt.show()

> 💡 **คำอธิบายผลลัพธ์** แต่ละจุดแทนผู้ป่วยหนึ่งราย สีแสดงการวินิจฉัยจริง (แดง = เป็นโรค, น้ำเงิน = ไม่เป็นโรค) ซึ่งแสดงไว้*เพื่อการพิจารณาของผู้เรียนเท่านั้น* PCA มิได้ใช้ข้อมูลนี้ หากจุดสีแดงและสีน้ำเงินอยู่ต่างบริเวณกัน แสดงว่าตัวแปรต่าง ๆ มี "รูปแบบของโรค" อยู่แล้วในตัว

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: การจัดกลุ่มด้วย K-means
# K-means ต้องกำหนดจำนวนกลุ่ม (clusters) ที่ต้องการ
# สิ่งที่ต้องทำ: แทนที่ `...` ด้วยเลข 2   (บรรทัดจะเป็น  K = 2 )
# ในคำถาม B2 ให้กลับมาทดลอง K = 3 แล้วรันเซลล์นี้และเซลล์โปรไฟล์ด้านล่างอีกครั้ง
K = ...   # TODO: จำนวนกลุ่ม

km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xs)
clusters = km.labels_          # หมายเลขกลุ่ม (0, 1, ...) ของผู้ป่วยแต่ละราย

plt.scatter(pca[:,0], pca[:,1], c=clusters, cmap="viridis", alpha=0.7)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title(f"K-means clusters (K={K})"); plt.show()

print("แถว = กลุ่ม (cluster), คอลัมน์ = การวินิจฉัยจริง (0 = ไม่เป็นโรค, 1 = เป็นโรค)")
print(pd.crosstab(clusters, y))

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
K = 2

km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xs)
clusters = km.labels_

plt.scatter(pca[:,0], pca[:,1], c=clusters, cmap="viridis", alpha=0.7)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title(f"K-means clusters (K={K})"); plt.show()

print("แถว = กลุ่ม (cluster), คอลัมน์ = การวินิจฉัยจริง (0 = ไม่เป็นโรค, 1 = เป็นโรค)")
print(pd.crosstab(clusters, y))

> 💡 **คำอธิบายผลลัพธ์** K-means จัดกลุ่มผู้ป่วยโดยใช้เฉพาะตัวแปรต่าง ๆ จากนั้นตารางไขว้ (cross-table) เปรียบเทียบกลุ่มที่ได้กับการวินิจฉัยจริง หมายเลขกลุ่ม (0, 1, …) เป็นเพียงชื่อเรียกที่กำหนดขึ้นโดยพลการ "กลุ่ม 1" มิได้ "รุนแรงกว่า" "กลุ่ม 0" โดยนิยาม

ขั้นต่อไปคือการบรรยายลักษณะของแต่ละกลุ่มในเชิงคลินิก โดยพิจารณาโปรไฟล์เฉลี่ยของผู้ป่วยในกลุ่ม

In [ ]:
# โปรไฟล์เฉลี่ยของผู้ป่วยในแต่ละกลุ่ม
profile = df.assign(cluster=clusters).groupby("cluster").agg(
    n_patients=("age", "size"),
    age=("age", "mean"),
    male_pct=("sex", "mean"),
    max_heart_rate=("thalach", "mean"),
    st_depression=("oldpeak", "mean"),
    exercise_angina_pct=("exang", "mean"),
    vessels_ca=("ca", "mean"),
    asymptomatic_cp_pct=("cp", lambda s: (s == 4).mean()),
    reversible_thal_pct=("thal", lambda s: (s == 7).mean()),
    disease_rate=("target", "mean"),
)
profile.round(2)

> 💡 **คำอธิบายผลลัพธ์** แต่ละแถวแทนหนึ่งกลุ่ม ความหมายของคอลัมน์มีดังนี้ `n_patients` = จำนวนผู้ป่วย, `male_pct` = สัดส่วนเพศชาย, `max_heart_rate` = ค่าเฉลี่ยของ `thalach`, `st_depression` = ค่าเฉลี่ยของ `oldpeak`, `exercise_angina_pct` = สัดส่วนผู้ที่มีอาการเจ็บหน้าอกขณะออกกำลังกาย, `vessels_ca` = ค่าเฉลี่ยของ `ca`, `asymptomatic_cp_pct` = สัดส่วนผู้ที่มี `cp` = 4, `reversible_thal_pct` = สัดส่วนผู้ที่มี `thal` = 7 คอลัมน์ `_pct` และ `disease_rate` (อัตราการเป็นโรค) แสดงเป็นสัดส่วน (0.80 = ร้อยละ 80) โดย `disease_rate` ใช้ป้ายกำกับ*ภายหลัง*การจัดกลุ่มเท่านั้น เพื่อตรวจสอบความหมายของกลุ่ม

✍️ **คำถาม B1** โปรดตั้งชื่อเชิงคลินิกโดยสังเขปให้แต่ละกลุ่มโดยอ้างอิงตารางโปรไฟล์ (เช่น "อายุน้อยกว่า สมรรถภาพการออกกำลังกายดี ความเสี่ยงต่ำ" และ "…") และจากตารางไขว้ กลุ่มที่ได้สอดคล้องกับการวินิจฉัยจริงมากน้อยเพียงใด

✍️ **คำตอบ:**

✍️ **คำถาม B2** ให้กำหนด `K = 3` ในแบบฝึกหัดที่ 3 แล้วรันเซลล์ดังกล่าวและเซลล์โปรไฟล์อีกครั้ง กลุ่มที่เพิ่มขึ้นมีลักษณะอย่างไร (คำแนะนำ: พิจารณา `male_pct`) จงอธิบายว่าแพทย์สามารถใช้ประโยชน์จากกลุ่มที่ได้จากการเรียนรู้แบบไม่มีผู้สอนได้อย่างไร และมีข้อควรระวังใดบ้าง (เช่น กลุ่มถูกค้นพบโดยไม่ได้ใช้ผลลัพธ์ทางคลินิก และค่า K ถูกกำหนดโดยผู้วิเคราะห์) เมื่อเสร็จแล้วให้กำหนด `K = 2` ดังเดิม (หรือรันเซลล์เฉลย) ก่อนดำเนินการต่อ

✍️ **คำตอบ:**

## Part C — การเรียนรู้แบบเสริมกำลัง (reinforcement learning, RL) (ตัวอย่างเชิงแนวคิด)

**การเรียนรู้แบบเสริมกำลัง** มุ่งเรียนรู้*นโยบาย (policy)* ว่าควรเลือก**การกระทำ (action)** ใดใน**สถานะ (state)** ต่าง ๆ เพื่อให้ได้**รางวัล (reward)** ระยะยาวสูงสุด โดยอาศัยการลองผิดลองถูก ในทางการแพทย์ เทียบได้กับการตัดสินใจรักษาที่ต่อเนื่องเป็นลำดับ (เช่น การปรับขนาดอินซูลิน การดูแลผู้ป่วยภาวะ sepsis)

ส่วนนี้ใช้สภาพแวดล้อม "การรักษา" จำลองขนาดเล็ก (ไม่ต้องใช้ไลบรารีภายนอก) ดังนี้
- **สถานะ** = ระดับความรุนแรงของผู้ป่วย 0–4 (0 = หายเป็นปกติ, 4 = วิกฤต)
- **การกระทำ** = ไม่รักษา (ผู้ป่วยอาจมีอาการแย่ลง) หรือรักษา (ผู้ป่วยมักมีอาการดีขึ้น)
- **รางวัล** = +10 เมื่อผู้ป่วยหายเป็นปกติ และมี**ต้นทุน (cost)** ทุกครั้งที่ให้การรักษา (ผลข้างเคียง ค่าใช้จ่าย)

เอเจนต์ (agent) ใช้วิธี **Q-learning** โดยเก็บตาราง `Q[ความรุนแรง, การกระทำ]` = "รางวัลในอนาคตที่คาดว่าจะได้รับ หากเลือกการกระทำนี้ในสถานะนี้" และปรับปรุงค่าหลังการจำลองทุกขั้น เมื่อจำลองผู้ป่วยครบ 3,000 ราย เอเจนต์จะเรียนรู้ว่าการกระทำใดให้ผลตอบแทนคุ้มค่า

In [ ]:
# สภาพแวดล้อมจำลองและเอเจนต์ Q-learning (สาระสำคัญอยู่ที่แนวคิด มิใช่รายละเอียดของโค้ด)
ACTION_NAMES = ["ไม่รักษา", "รักษา"]

def train_agent(treat_cost=-1, sickness_penalty=0, episodes=3000, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((5, 2))                      # แถว = ความรุนแรง 0..4, คอลัมน์ = [ไม่รักษา, รักษา]

    def step(s, a):
        if a == 1:                            # รักษา: ความรุนแรงมักลดลง 1-2 ระดับ
            s2 = max(0, s - rng.choice([1, 1, 2]))
            r = treat_cost                    # ต้นทุนของการรักษา
        else:                                 # ไม่รักษา: อาการอาจแย่ลง
            s2 = min(4, s + rng.choice([0, 0, 1]))
            r = 0
        r -= sickness_penalty * s2            # ไม่บังคับ: บทลงโทษทุกขั้นที่ผู้ป่วยยังมีอาการ
        if s2 == 0:
            r += 10                           # รางวัลเมื่อผู้ป่วยหายเป็นปกติ
        return s2, r

    alpha, gamma, eps = 0.1, 0.9, 0.2         # อัตราการเรียนรู้, ตัวลดค่ารางวัลในอนาคต, อัตราการสำรวจ (exploration)
    for episode in range(episodes):
        s = rng.integers(1, 5)                # ผู้ป่วยรายใหม่ ความรุนแรง 1-4
        for _ in range(20):                   # ตัดสินใจได้สูงสุด 20 ขั้น
            a = rng.integers(2) if rng.random() < eps else int(np.argmax(Q[s]))
            s2, r = step(s, a)
            Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
            s = s2
            if s == 0: break                  # หายเป็นปกติ -> สิ้นสุดรอบ (episode)
    return Q

def show_policy(Q, title):
    print(title)
    print("ตาราง Q (แถว = ความรุนแรง, คอลัมน์ = [ไม่รักษา, รักษา]):")
    print(np.round(Q, 1))
    for s in range(1, 5):                     # ความรุนแรง 0 = หายเป็นปกติ = สิ้นสุด ไม่ต้องตัดสินใจ
        print(f"  ความรุนแรง {s}: การกระทำที่ดีที่สุด = {ACTION_NAMES[int(np.argmax(Q[s]))]}")

Q = train_agent(treat_cost=-1)
show_policy(Q, "ต้นทุนการรักษา = -1")

> 💡 **คำอธิบายผลลัพธ์** เอเจนต์ไม่ได้รับกฎทางการแพทย์ใด ๆ แต่เรียนรู้จากรางวัลเพียงอย่างเดียว ตาราง Q แสดงผลตอบแทนของแต่ละการกระทำในแต่ละระดับความรุนแรง การกระทำที่ดีที่สุดคือการกระทำที่มีค่ามากกว่าในแถวนั้น แถว 0 (หายเป็นปกติ) มีค่าเป็น 0 เนื่องจากรอบสิ้นสุด ณ สถานะนั้น

In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: ปรับรางวัลและพิจารณาสิ่งที่เอเจนต์เรียนรู้
# สิ่งที่ต้องทำ: เพิ่มต้นทุนการรักษา โดยแทนที่ `...` ด้วย  -5
# จากนั้นรันเซลล์ และเปรียบเทียบนโยบายที่ได้กับผลข้างต้น (ต้นทุน -1)
# ไม่บังคับ: หลังจากนั้นให้ทดลองกำหนด SICKNESS_PENALTY = 1 (บทลงโทษทุกขั้นที่ผู้ป่วยยังมีอาการ)
# แล้วรันเซลล์อีกครั้ง
TREAT_COST = ...          # TODO: ทดลอง -5
SICKNESS_PENALTY = 0      # ไม่บังคับ: ทดลอง 1

Q_new = train_agent(treat_cost=TREAT_COST, sickness_penalty=SICKNESS_PENALTY)
show_policy(Q_new, f"ต้นทุนการรักษา = {TREAT_COST}, บทลงโทษเมื่อยังมีอาการ = {SICKNESS_PENALTY}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
TREAT_COST = -5
SICKNESS_PENALTY = 0

Q_new = train_agent(treat_cost=TREAT_COST, sickness_penalty=SICKNESS_PENALTY)
show_policy(Q_new, f"ต้นทุนการรักษา = {TREAT_COST}, บทลงโทษเมื่อยังมีอาการ = {SICKNESS_PENALTY}")

print()
show_policy(train_agent(treat_cost=-5, sickness_penalty=1),
            "ต้นทุนเท่าเดิม (-5) และเพิ่มบทลงโทษเมื่อยังมีอาการ 1 ต่อขั้น")

✍️ **คำถาม C1** เมื่อต้นทุนการรักษาเท่ากับ −1 เอเจนต์เรียนรู้ที่จะให้การรักษาผู้ป่วยในระดับความรุนแรงใดบ้าง และนโยบายดังกล่าวสมเหตุสมผลทางคลินิกหรือไม่

✍️ **คำตอบ:**

✍️ **คำถาม C2** เมื่อต้นทุนเท่ากับ −5 เอเจนต์**ไม่ให้การรักษา**ผู้ป่วยกลุ่มใด จงอธิบายว่าเหตุใดพฤติกรรมนี้จึง "มีเหตุผล" สำหรับเอเจนต์ภายใต้ฟังก์ชันรางวัลที่กำหนดไว้ (คำแนะนำ: พิจารณาว่ามีบทลงโทษต่อการที่ผู้ป่วยอยู่ในภาวะวิกฤตหรือไม่) เมื่อเพิ่มบทลงโทษเมื่อยังมีอาการแล้ว ผลเปลี่ยนแปลงอย่างไร และประเด็นนี้ให้ข้อคิดใดต่อการออกแบบรางวัล (reward design) สำหรับระบบปัญญาประดิษฐ์ (AI) ที่ใช้แนะนำการรักษาในทางปฏิบัติ

✍️ **คำตอบ:**

## สรุป — การเลือกรูปแบบการเรียนรู้ให้เหมาะสมกับโจทย์

✍️ **คำถาม D1 (คำถามหลัก 2–3 ประโยค)** สำหรับโจทย์ด้านสุขภาพดิจิทัล (digital health) ที่ผู้เรียนคุ้นเคยจากการปฏิบัติงาน ผู้เรียนจะเลือกใช้การเรียนรู้แบบ**มีผู้สอน** แบบ**ไม่มีผู้สอน** และแบบ**เสริมกำลัง**ในกรณีใด โปรดยกตัวอย่างที่เป็นรูปธรรมอย่างละ 1 ตัวอย่าง (เช่น "การทำนายการกลับเข้ารับการรักษาซ้ำในโรงพยาบาลภายใน 30 วันจากข้อมูลเมื่อจำหน่าย → การเรียนรู้แบบมีผู้สอน เนื่องจาก …")

✍️ **คำตอบ:**

## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ไม่บังคับ และไม่นับเป็นชิ้นงานที่ต้องส่ง** ผู้เรียนทุกคน*สามารถ*ส่งผลได้ เนื่องจากโมเดลพื้นฐาน (baseline) ด้านล่างใช้งานได้ทันที (ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่า แล้วรันเซลล์นั้นอีกครั้งก่อน) ผู้เรียนที่เขียนโปรแกรมได้สามารถพัฒนาโมเดลให้ได้คะแนนสูงกว่า baseline

**โจทย์: การประเมินความเสี่ยงโรคหัวใจ** ให้ทำนายความน่าจะเป็นของการเป็นโรคหัวใจสำหรับ**ผู้ป่วย 74 รายที่แยกไว้ (held-out)** (รหัสอยู่ในไฟล์ `heart/test_ids.csv` โดยรหัสคือหมายเลขแถวใน `df`) **ตัวชี้วัด: ROC AUC** การแบ่งข้อมูลข้างต้น (`X_tr`/`X_te`) ใช้เพื่อการเรียนรู้เท่านั้น สำหรับตารางอันดับให้ฝึกโมเดลใหม่ด้วย**ข้อมูลทุกแถว ยกเว้นรหัสผู้ป่วยชุดทดสอบของตารางอันดับ** และเลือกการตั้งค่าด้วยการตรวจสอบไขว้บนแถวฝึกดังกล่าวเท่านั้น

**ไฟล์ที่ส่ง:** `lab03_submission.csv` มีหัวตาราง `id,prob` หนึ่งแถวต่อรหัสผู้ป่วยชุดทดสอบหนึ่งราย โดย `prob` = ความน่าจะเป็นของการเป็นโรค P(disease)

> **จรรยาบรรณทางวิชาการ (honour code)** ป้ายกำกับที่ซ่อนไว้มาจากชุดข้อมูลสาธารณะ การค้นหาคำตอบจากแหล่งภายนอกจึงทำได้ง่ายและไม่เกิดประโยชน์ต่อการเรียนรู้ ตารางอันดับจัดทำขึ้นเพื่อสนับสนุนการเรียนรู้ และโน้ตบุ๊กของผู้เรียนจะได้รับการประเมินด้วย ห้ามแก้ไขผลการทำนายด้วยมือโดยเด็ดขาด

In [ ]:
# 🏆 Baseline: random forest ที่ฝึกใหม่ด้วยทุกแถวซึ่งไม่อยู่ในชุดทดสอบ -> lab03_submission.csv
test_ids = pd.read_csv(lab_data("heart/test_ids.csv"))["id"]
train_mask = ~X.index.isin(test_ids)
X_lb, y_lb = X[train_mask], y[train_mask]

def make_submission(model, path="lab03_submission.csv"):
    cv = cross_val_score(model, X_lb, y_lb, cv=5, scoring="roc_auc").mean()
    print(f"CV ROC AUC บนผู้ป่วยที่ไม่อยู่ในชุดทดสอบ {len(X_lb)} ราย: {cv:.3f}")
    model.fit(X_lb, y_lb)
    prob = model.predict_proba(X.loc[test_ids])[:, 1]
    pd.DataFrame({"id": test_ids.values, "prob": prob}).to_csv(path, index=False)
    print(f"บันทึกไฟล์ {path} เรียบร้อย ({len(prob)} แถว)")
    if NAME and JOIN_CODE:
        submit_to_leaderboard(path)
    else:
        print("โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า (และรันเซลล์นั้นอีกครั้ง) เพื่อส่งผลไปยังตารางอันดับ")

make_submission(RandomForestClassifier(n_estimators=300, random_state=42))

### แนวทางการพัฒนาให้ได้คะแนนสูงกว่า baseline
- **ปรับค่าไฮเปอร์พารามิเตอร์ (hyperparameter tuning)** ของต้นไม้ ได้แก่ `max_depth`, `min_samples_leaf`, `n_estimators` และสำหรับ XGBoost ได้แก่ `learning_rate`, `subsample`, `colsample_bytree` (เช่น ใช้ `GridSearchCV` บน `X_lb`, `y_lb`)
- **เข้ารหัสตัวแปรหมวดหมู่ให้เหมาะสม:** ใช้ one-hot encoding กับ `cp`, `restecg`, `slope`, `thal` (เช่น `pd.get_dummies`) ซึ่งเป็นประโยชน์อย่างยิ่งต่อ logistic regression
- **ใช้ regularisation** กับ logistic regression (`C` ระหว่าง 0.01 ถึง 1) เนื่องจากโมเดลที่เรียบง่ายมักให้ผลดีกับข้อมูลผู้ป่วยประมาณ 300 ราย
- **Ensemble:** เฉลี่ยความน่าจะเป็นจากหลายโมเดล (`VotingClassifier(voting="soft")`)
- ควรให้น้ำหนักกับคะแนน CV มากกว่าคะแนนบนตารางอันดับ เนื่องจากชุดทดสอบมีผู้ป่วยเพียง 74 ราย ค่า AUC บนตารางอันดับจึงมีความแปรปรวนสูง

In [ ]:
# 🏆 ไม่บังคับ: โมเดลที่ผู้เรียนพัฒนาเอง ให้แทนที่ตัวอย่างด้านล่างด้วยโมเดลของผู้เรียน แล้วรันเซลล์
from sklearn.ensemble import VotingClassifier

my_model = VotingClassifier(
    [("lr", make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=1000))),
     ("rf", RandomForestClassifier(n_estimators=500, min_samples_leaf=3, random_state=42))],
    voting="soft",
)
make_submission(my_model)

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่งโน้ตบุ๊กนี้ที่รันครบทุกเซลล์ตามลำดับจากบนลงล่าง พร้อมคำตอบในเซลล์ ✍️ **คำตอบ** ทุกเซลล์ (A1–A3, B1–B2, C1–C2 และ D1 โดย D1 เป็นคำถามหลัก) ตามช่องทางที่กำหนดในแนวปฏิบัติของรายวิชา

ผู้เรียนที่เข้าร่วมโจทย์ท้าทาย 🏆 บนตารางอันดับ สามารถระบุชื่อและคะแนนสูงสุดบนตารางอันดับไว้ในส่วนนี้ (ไม่บังคับ และไม่นำมาคิดคะแนน)

✍️ **ชื่อ / คะแนนบนตารางอันดับ (ไม่บังคับ):**